---
## 4. End-to-end smoke test

Synthetic pipeline:
1. Create a simple sharp test image (geometric shapes — known ground truth)
2. Apply known Gaussian blur + noise → blurry observation $\mathbf{f}$
3. Run alternating minimization on $\mathbf{f}$
4. Verify constraints are satisfied and objective decreases
5. Visualize each stage

> **Prototype scale note:** `build_Au_from_k` materialises $H_k \in \mathbb{R}^{N \times N}$.
> For a $32 \times 32$ image that is a $1024 \times 1024$ dense matrix fed into CVXPY's
> conic solver (SCS). SCS is a general-purpose solver — it satisfies all constraints
> and decreases the objective, but numerical precision degrades on large ill-conditioned
> systems. **Visual quality improvement requires replacing `build_Au_from_k` with the
> FFT-based solve** (next notebook), which keeps the problem in frequency domain
> and avoids materialising $H_k$ entirely.
>
> The purpose of this notebook is to verify the mathematical model is correct —
> constraints satisfied, objective monotone, kernel sum = 1, pixels in $[0,255]$.
> Treat PSNR here as a sanity check, not a performance benchmark.


## 1. Imports and hyperparameters

In [1]:
pip install cvxpy

  Using cached numpy-2.4.4-cp312-cp312-win_amd64.whl.metadata (6.6 kB)
  Using cached numpy-2.2.6-cp312-cp312-win_amd64.whl.metadata (60 kB)
Using cached numpy-2.2.6-cp312-cp312-win_amd64.whl (12.6 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
Note: you may need to restart the kernel to use updated packages.


  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
contourpy 1.2.0 requires numpy<2.0,>=1.20, but you have numpy 2.2.6 which is incompatible.
gensim 4.3.3 requires numpy<2.0,>=1.18.5, but you have numpy 2.2.6 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.2.6 which is incompatible.


In [2]:
pip install "numpy<2"

  Using cached numpy-1.26.4-cp312-cp312-win_amd64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp312-cp312-win_amd64.whl (15.5 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.6
    Uninstalling numpy-2.2.6:
      Successfully uninstalled numpy-2.2.6
Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cvxpy 1.8.2 requires numpy>=2.0.0, but you have numpy 1.26.4 which is incompatible.


In [3]:
import numpy as np
import cvxpy as cp
from scipy.ndimage import convolve
from scipy.linalg import norm
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# ── Hyperparameters (tune these) ──────────────────────────────────────────
LAMBDA1      = 0.1    # TV weight        — controls edge sharpness
LAMBDA2      = 0.05   # Tikhonov weight  — controls smoothness / stability
MU           = 0.01   # kernel L1 weight — controls kernel sparsity
KERNEL_SIZE  = 5      # side length of blur kernel to estimate
MAX_ITER     = 6      # alternating minimization iterations
TOL          = 1e-4   # convergence tolerance

print("Imports OK. Hyperparameters set.")

ModuleNotFoundError: No module named 'numpy.lib.array_utils'

## 2. Math model

Each function below corresponds directly to a term in the mathematical formulation.


### 2a. Gradient operator $D : \mathbb{R}^N \to \mathbb{R}^{2N}$

Discrete forward finite differences:
$$
(D\mathbf{u})_i = \begin{pmatrix} u_{i,j+1} - u_{i,j} \\ u_{i+1,j} - u_{i,j} \end{pmatrix}
$$

Boundary pixels use zero-padding (Neumann boundary condition).


In [ ]:
def gradient(u: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """
    Discrete image gradient via forward finite differences.
    Returns dx (horizontal) and dy (vertical), both shape (H, W).
    """
    dx = np.zeros_like(u)
    dy = np.zeros_like(u)
    dx[:, :-1] = u[:, 1:] - u[:, :-1]   # partial_x
    dy[:-1, :] = u[1:, :] - u[:-1, :]   # partial_y
    return dx, dy

print("gradient() defined.")


### 2b. Total Variation — $\|\nabla \mathbf{u}\|_1$ (isotropic, mixed $\ell_{1,2}$ norm)

$$
\text{TV}(\mathbf{u}) = \sum_{i=1}^{N} \|(D\mathbf{u})_i\|_2
= \sum_{i,j} \sqrt{(\partial_x u_{ij})^2 + (\partial_y u_{ij})^2}
$$

**Why this promotes sharp edges:** the penalty grows *linearly* in gradient magnitude,
so the optimizer concentrates all gradient budget into a few pixels (edges) and sets
the rest to zero — piecewise-constant image with sparse sharp boundaries.


In [ ]:
def tv_norm(u: np.ndarray) -> float:
    """
    Isotropic Total Variation: sum_i ||(Du)_i||_2
    Mixed l_{1,2} norm. Rotation-invariant. Promotes piecewise-constant images.
    """
    dx, dy = gradient(u)
    return float(np.sum(np.sqrt(dx**2 + dy**2)))

print("tv_norm() defined.")


### 2c. Tikhonov regularization — $\|D\mathbf{u}\|_2^2$

$$
\|D\mathbf{u}\|_2^2 = \sum_{i,j} (\partial_x u_{ij})^2 + (\partial_y u_{ij})^2
= \mathbf{u}^T D^T D \mathbf{u}
$$

**Role:** quadratic in $\mathbf{u}$, so it adds positive curvature $\lambda_2 D^T D$
to the Hessian of the data fidelity term. This shifts all eigenvalues away from zero,
fixing the ill-conditioning introduced by the blur. Also prevents TV staircase artifacts
in smooth regions by keeping small gradient differences alive.


In [ ]:
def tikhonov_norm_sq(u: np.ndarray) -> float:
    """
    ||D u||_2^2 = sum_{i,j} dx^2 + dy^2  (quadratic form u^T D^T D u).
    Stabilises the ill-conditioned normal equations from blur inversion.
    """
    dx, dy = gradient(u)
    return float(np.sum(dx**2 + dy**2))

print("tikhonov_norm_sq() defined.")


### 2d. Blur operator $H$ and the matrix $A_\mathbf{u}$

The forward model is $\mathbf{f} = k * \mathbf{u} + \mathbf{n}$.

- `apply_blur(u, k)` computes $H\mathbf{u} = k * \mathbf{u}$
- `build_Au(u, kernel_size)` builds $A_\mathbf{u}$ such that $A_\mathbf{u} k = k * \mathbf{u}$
  — the key identity that makes the k-subproblem a **LASSO**
- `build_Au_from_k(k, H, W)` builds $H_k$ such that $H_k \mathbf{u} = k * \mathbf{u}$
  — used in the u-subproblem


In [ ]:
def apply_blur(u: np.ndarray, k: np.ndarray) -> np.ndarray:
    """H u = k * u via convolution (reflect boundary)."""
    return convolve(u, k, mode='reflect')


def build_Au(u: np.ndarray, kernel_size: int) -> np.ndarray:
    """
    Build A_u (shape N x m) so that  A_u @ k.ravel() == (k * u).ravel().
    Convolution by u is LINEAR in k when u is fixed.
    Each column j = image convolved with j-th basis kernel.
    """
    H, W = u.shape
    m    = kernel_size ** 2
    N    = H * W
    A    = np.zeros((N, m))
    for j in range(m):
        e_j         = np.zeros(m);  e_j[j] = 1.0
        k_basis     = e_j.reshape(kernel_size, kernel_size)
        A[:, j]     = convolve(u, k_basis, mode='reflect').ravel()
    return A


def build_Au_from_k(k: np.ndarray, H_im: int, W_im: int) -> np.ndarray:
    """
    Build H_k (shape N x N) so that  H_k @ u.ravel('F') == (k * u).ravel('F').
    NOTE: for large images this is memory-heavy — fine for prototype scale.
    Production code replaces this with the FFT diagonalisation trick.
    """
    N  = H_im * W_im
    A  = np.zeros((N, N))
    ej = np.zeros(N)
    for j in range(N):
        ej[:] = 0.0;  ej[j] = 1.0
        img_j    = ej.reshape(H_im, W_im, order='F')
        A[:, j]  = convolve(img_j, k, mode='reflect').ravel(order='F')
    return A

print("apply_blur(), build_Au(), build_Au_from_k() defined.")


### 2e. Data fidelity and joint objective

$$
J(k, \mathbf{u}) = \|k * \mathbf{u} - \mathbf{f}\|_2^2
+ \lambda_1 \text{TV}(\mathbf{u})
+ \lambda_2 \|D\mathbf{u}\|_2^2
+ \mu \|k\|_1
$$

`joint_objective` evaluates all four terms separately — useful for monitoring
which term dominates during alternating minimization iterations.


In [ ]:
def data_fidelity(u, k, f) -> float:
    """
    ||k * u - f||_2^2.
    Convex quadratic in u (H fixed) or in k (u fixed). Bilinear jointly => non-convex.
    Derived from MLE under i.i.d. Gaussian noise n ~ N(0, sigma^2 I).
    """
    return float(np.sum((apply_blur(u, k) - f) ** 2))


def joint_objective(u, k, f, lambda1, lambda2, mu) -> dict:
    """Evaluate and return all four terms of the joint objective."""
    fid  = data_fidelity(u, k, f)
    tv   = lambda1 * tv_norm(u)
    tikh = lambda2 * tikhonov_norm_sq(u)
    kern = mu * float(np.sum(np.abs(k)))
    return {
        "data_fidelity":           fid,
        "tv_regularization":       tv,
        "tikhonov_regularization": tikh,
        "kernel_l1":               kern,
        "total":                   fid + tv + tikh + kern,
    }

print("data_fidelity(), joint_objective() defined.")


### 2f. k-subproblem — constrained LASSO

Fix $\mathbf{u}^{(t)}$. With $\mathbf{u}$ fixed, convolution is **linear in $k$**,
so the subproblem becomes:

$$
k^{(t+1)} = \arg\min_{k \geq 0,\; \mathbf{1}^T k = 1}
\; \|A_{\mathbf{u}^{(t)}} k - \mathbf{f}\|_2^2 + \mu \|k\|_1
$$

This is exactly **L2 optimization with L1 regularization** (LASSO) from lecture,
with the added simplex constraint $k \in \Delta^m$.  
Solved by CVXPY → SCS.


In [ ]:
def solve_k_subproblem(u, f, kernel_size, mu, verbose=False):
    """
    k-subproblem: constrained LASSO.
    min_k  ||A_u k - f||^2 + mu ||k||_1
    s.t.   k >= 0,  sum(k) = 1
    """
    m   = kernel_size ** 2
    A   = build_Au(u, kernel_size)   # shape (N, m)
    f_v = f.ravel()

    k_var = cp.Variable(m, name="k")
    obj   = cp.Minimize(cp.sum_squares(A @ k_var - f_v) + mu * cp.norm1(k_var))
    cons  = [k_var >= 0, cp.sum(k_var) == 1]
    prob  = cp.Problem(obj, cons)
    prob.solve(solver=cp.SCS, verbose=verbose)

    if prob.status not in ("optimal", "optimal_inaccurate"):
        raise RuntimeError(f"k-subproblem: {prob.status}")
    return k_var.value.reshape(kernel_size, kernel_size)

print("solve_k_subproblem() defined.")


### 2g. u-subproblem — TV + Tikhonov deconvolution

Fix $k^{(t+1)}$. Build $H^{(t+1)}$ from it. The subproblem is **convex in $\mathbf{u}$**:

$$
\mathbf{u}^{(t+1)} = \arg\min_{0 \leq \mathbf{u} \leq 255}
\; \|H^{(t+1)}\mathbf{u} - \mathbf{f}\|_2^2
+ \lambda_1 \text{TV}(\mathbf{u})
+ \lambda_2 \|D\mathbf{u}\|_2^2
$$

CVXPY uses `cp.tv()` for the isotropic TV atom.  
Tikhonov uses explicit finite differences.  
Solved by CVXPY → SCS.


In [ ]:
def solve_u_subproblem(k, f, lambda1, lambda2, verbose=False):
    """
    u-subproblem: TV + Tikhonov deconvolution.
    min_u  ||H u - f||^2 + lambda1 TV(u) + lambda2 ||Du||^2
    s.t.   0 <= u <= 255
    """
    H_im, W_im = f.shape
    A_k = build_Au_from_k(k, H_im, W_im)   # blur matrix, shape (N, N)
    f_v = f.ravel(order='F')

    u_var = cp.Variable((H_im, W_im), name="u")
    u_v   = cp.vec(u_var, order='F')

    data  = cp.sum_squares(A_k @ u_v - f_v)
    tv    = lambda1 * cp.tv(u_var)
    dx    = u_var[:, 1:] - u_var[:, :-1]
    dy    = u_var[1:, :] - u_var[:-1, :]
    tikh  = lambda2 * (cp.sum_squares(dx) + cp.sum_squares(dy))

    prob  = cp.Problem(cp.Minimize(data + tv + tikh), [u_var >= 0, u_var <= 255])
    prob.solve(solver=cp.SCS, verbose=verbose)

    if prob.status not in ("optimal", "optimal_inaccurate"):
        raise RuntimeError(f"u-subproblem: {prob.status}")
    return u_var.value

print("solve_u_subproblem() defined.")


### 2h. Utilities — Gaussian kernel, synthetic blur, PSNR

In [ ]:
def _gaussian_kernel(size: int, sigma: float) -> np.ndarray:
    """
    Normalized 2D Gaussian kernel: k^(0) initialization.
    Satisfies k >= 0 and 1^T k = 1 by construction.
    """
    ax = np.arange(size) - size // 2
    xx, yy = np.meshgrid(ax, ax)
    k = np.exp(-(xx**2 + yy**2) / (2 * sigma**2))
    return k / k.sum()


def make_synthetic_blur(u_sharp, kernel_size=5, sigma=1.5, noise_std=2.0):
    """
    Degrade a sharp image with known Gaussian blur + Gaussian noise.
    Returns (f, k_true) for ground-truth validation.
    """
    k_true  = _gaussian_kernel(kernel_size, sigma)
    f_clean = apply_blur(u_sharp.astype(float), k_true)
    f       = np.clip(f_clean + np.random.normal(0, noise_std, f_clean.shape), 0, 255)
    return f, k_true


def psnr(u_true, u_hat, max_val=255.0) -> float:
    """
    Peak Signal-to-Noise Ratio (dB).
    PSNR = 10 log10(max_val^2 / MSE).  Higher is better; >30 dB = good quality.
    """
    mse = np.mean((u_true.astype(float) - u_hat.astype(float)) ** 2)
    return float('inf') if mse == 0 else 10 * np.log10(max_val**2 / mse)

print("_gaussian_kernel(), make_synthetic_blur(), psnr() defined.")


### 2i. Alternating minimization main loop

$$
\text{Initialize: } k^{(0)} = \text{Gaussian}, \quad \mathbf{u}^{(0)} = \mathbf{f}
$$

$$
\text{For } t = 0, 1, 2, \ldots:\quad
k^{(t+1)} \leftarrow \text{k-step},\quad
\mathbf{u}^{(t+1)} \leftarrow \text{u-step},\quad
\text{check convergence}
$$

**Convergence criterion:**
$$
\frac{\|k^{(t+1)} - k^{(t)}\|_2}{\|k^{(t)}\|_2} < \epsilon_k
\quad \text{and} \quad
\frac{\|\mathbf{u}^{(t+1)} - \mathbf{u}^{(t)}\|_2}{\|\mathbf{u}^{(t)}\|_2} < \epsilon_u
$$


In [ ]:
def alternating_minimization(f, kernel_size=5, lambda1=0.1, lambda2=0.05,
                              mu=0.01, max_iter=10, tol_k=1e-4, tol_u=1e-4,
                              verbose=True):
    """
    Blind image deconvolution via alternating minimization.
    Each iteration solves the convex k-subproblem then the convex u-subproblem.
    """
    k = _gaussian_kernel(kernel_size, sigma=1.0)   # k^(0)
    u = f.copy().astype(float)                     # u^(0) = f

    history = []

    if verbose:
        header = f"{'Iter':>4}  {'Total':>12}  {'Fidelity':>12}  {'TV':>10}  {'Tikhonov':>10}  {'KernelL1':>10}  {'dk':>9}  {'du':>9}"
        print(header)
        print("-" * len(header))

    for t in range(max_iter):
        k_prev, u_prev = k.copy(), u.copy()

        k = solve_k_subproblem(u, f, kernel_size, mu)
        u = solve_u_subproblem(k, f, lambda1, lambda2)

        obj = joint_objective(u, k, f, lambda1, lambda2, mu)
        history.append(obj)

        dk = norm(k - k_prev) / (norm(k_prev) + 1e-10)
        du = norm(u - u_prev) / (norm(u_prev) + 1e-10)

        if verbose:
            print(f"{t+1:>4}  {obj['total']:>12.4f}  {obj['data_fidelity']:>12.4f}  "
                  f"{obj['tv_regularization']:>10.4f}  {obj['tikhonov_regularization']:>10.4f}  "
                  f"{obj['kernel_l1']:>10.4f}  {dk:>9.2e}  {du:>9.2e}")

        if dk < tol_k and du < tol_u:
            if verbose: print(f"\nConverged at iteration {t+1}.")
            break
    else:
        if verbose: print(f"\nMax iterations ({max_iter}) reached.")

    return {"u_hat": u, "k_hat": k, "history": history}

print("alternating_minimization() defined.")
print()
print("All model functions loaded. Proceed to tests.")


---
## 3. Unit tests

Each test maps to a specific mathematical claim.  
Run this cell — all assertions must pass before proceeding to the end-to-end test.


### 3a. Gradient operator

In [ ]:
# Constant image => zero gradient
u_flat = np.ones((8, 8)) * 128.0
dx, dy = gradient(u_flat)
assert np.allclose(dx, 0) and np.allclose(dy, 0), "FAIL: constant image should have zero gradient"
print("PASS  constant image has zero gradient")

# Shape preserved
dx, dy = gradient(np.random.rand(10, 12))
assert dx.shape == (10, 12) and dy.shape == (10, 12)
print("PASS  gradient shape matches image")

# Linearity: D(a*u + b*v) = a*D(u) + b*D(v)
u, v = np.random.rand(8, 8), np.random.rand(8, 8)
a, b = 3.0, -1.5
dx_u, dy_u = gradient(u);  dx_v, dy_v = gradient(v)
dx_c, dy_c = gradient(a*u + b*v)
assert np.allclose(dx_c, a*dx_u + b*dx_v) and np.allclose(dy_c, a*dy_u + b*dy_v)
print("PASS  gradient is linear: D(au+bv) = aDu + bDv")

# Boundary zero-padding
u_arange = np.arange(16, dtype=float).reshape(4, 4)
dx_a, dy_a = gradient(u_arange)
assert np.allclose(dx_a[:, -1], 0) and np.allclose(dy_a[-1, :], 0)
print("PASS  boundary pixels have zero gradient (Neumann BC)")


### 3b. TV norm

In [ ]:
# TV of constant = 0
assert tv_norm(np.ones((8,8)) * 50.0) == 0.0
print("PASS  TV(constant) = 0")

# TV >= 0
assert tv_norm(np.random.rand(10,10) * 255) >= 0
print("PASS  TV >= 0")

# TV(alpha * u) = alpha * TV(u)
u = np.random.rand(8, 8);  alpha = 3.7
assert abs(tv_norm(alpha * u) - alpha * tv_norm(u)) < 1e-10
print("PASS  TV(alpha*u) = alpha*TV(u)")

# Triangle inequality: TV(u+v) <= TV(u) + TV(v)
u, v = np.random.rand(8,8), np.random.rand(8,8)
assert tv_norm(u + v) <= tv_norm(u) + tv_norm(v) + 1e-10
print("PASS  TV triangle inequality")

# Exact value for step edge
H_s, W_s = 4, 6
u_step = np.zeros((H_s, W_s));  u_step[:, W_s//2:] = 1.0
# One vertical edge of height 1 running all 4 rows => TV = 4.0
assert abs(tv_norm(u_step) - float(H_s)) < 1e-10
print(f"PASS  step edge TV = {H_s} (exact analytic value)")


### 3c. Tikhonov norm

In [ ]:
# Constant => 0
assert tikhonov_norm_sq(np.ones((8,8)) * 100.0) == 0.0
print("PASS  Tikhonov(constant) = 0")

# >= 0
assert tikhonov_norm_sq(np.random.rand(10,10)) >= 0
print("PASS  Tikhonov >= 0")

# Quadratic scaling: ||D(alpha*u)||^2 = alpha^2 * ||Du||^2
u = np.random.rand(8, 8);  alpha = 2.5
assert abs(tikhonov_norm_sq(alpha*u) - alpha**2 * tikhonov_norm_sq(u)) < 1e-8
print("PASS  Tikhonov(alpha*u) = alpha^2 * Tikhonov(u)  [quadratic scaling]")


### 3d. Blur operator and $A_\mathbf{u}$ matrix identity

In [ ]:
# Delta kernel = identity
u = np.random.rand(10, 10) * 255
k_delta = np.zeros((5,5));  k_delta[2,2] = 1.0
assert np.allclose(apply_blur(u, k_delta), u, atol=1e-6)
print("PASS  delta kernel is identity")

# Normalized kernel preserves mean brightness
k_gauss = _gaussian_kernel(5, 1.0)
assert abs(k_gauss.sum() - 1.0) < 1e-10
u = np.random.rand(12, 12) * 255
assert abs(np.mean(apply_blur(u, k_gauss)) - np.mean(u)) < 0.5
print("PASS  normalized kernel preserves mean (energy conservation)")

# Blur is linear: H(au+bv) = aHu + bHv
u, v = np.random.rand(10,10), np.random.rand(10,10)
a, b = 2.0, -0.5
assert np.allclose(apply_blur(a*u+b*v, k_gauss), a*apply_blur(u,k_gauss)+b*apply_blur(v,k_gauss))
print("PASS  blur operator is linear")

# KEY IDENTITY: A_u @ k.ravel() == (k * u).ravel()
u = np.random.rand(8, 8)
k = _gaussian_kernel(3, 0.8)
A = build_Au(u, kernel_size=3)
assert np.allclose(A @ k.ravel(), apply_blur(u, k).ravel(), atol=1e-6)
print("PASS  A_u @ k = k * u  (k-subproblem becomes LASSO)")


### 3e. Gaussian kernel properties

In [ ]:
k = _gaussian_kernel(7, sigma=1.5)
assert abs(k.sum() - 1.0) < 1e-10;        print("PASS  sum(k) = 1  (simplex, energy conservation)")
assert np.all(k >= 0);                     print("PASS  k >= 0  (non-negative blur)")
assert np.allclose(k, k.T);               print("PASS  k is symmetric")
assert k[3, 3] == k.max();                print("PASS  peak at center")
for s in [3, 5, 7]:
    assert _gaussian_kernel(s, 1.0).shape == (s, s)
print("PASS  kernel shapes correct for sizes 3, 5, 7")


### 3f. Data fidelity

In [ ]:
# Zero fidelity when u is perfectly consistent with f (delta kernel)
u = np.random.rand(8, 8) * 255
k_d = np.zeros((3,3));  k_d[1,1] = 1.0
f = apply_blur(u, k_d)
assert data_fidelity(u, k_d, f) < 1e-6
print("PASS  data_fidelity = 0 when u exactly explains f")

# Always non-negative
k = _gaussian_kernel(3, 0.8)
f = np.random.rand(8,8)*255
assert data_fidelity(np.random.rand(8,8)*255, k, f) >= 0
print("PASS  data_fidelity >= 0")

# Increases with mismatch
u = np.random.rand(8,8)*255;  k = _gaussian_kernel(3,1.0);  f = apply_blur(u,k)
assert data_fidelity(u+np.random.normal(0,10,u.shape), k, f) > data_fidelity(u, k, f)
print("PASS  data_fidelity increases when u drifts from true image")


### 3g. PSNR

In [ ]:
u = np.random.rand(8,8)*255
assert psnr(u, u) == float('inf');                         print("PASS  PSNR(u, u) = inf")
u_hat = u + np.random.normal(0, 5, u.shape)
assert psnr(u, u_hat) > 0;                                 print("PASS  PSNR > 0")
u_base = np.ones((16,16))*128
assert psnr(u_base, u_base+np.random.normal(0,1,u_base.shape)) >        psnr(u_base, u_base+np.random.normal(0,20,u_base.shape))
print("PASS  PSNR decreases with more noise")


### 3h. Solver constraints (small-scale)

In [ ]:
import time

# k-subproblem: solved kernel must satisfy simplex constraints
print("Running k-subproblem test (10x10 image, 3x3 kernel)...")
t0 = time.time()
u_test = np.random.rand(10, 10) * 128
f_test, _ = make_synthetic_blur(u_test, kernel_size=3, sigma=1.0, noise_std=1.0)
k_hat = solve_k_subproblem(u_test, f_test, kernel_size=3, mu=0.01)
print(f"  solved in {time.time()-t0:.1f}s")
assert np.all(k_hat >= -1e-4),                    "k must be non-negative"
assert abs(k_hat.sum() - 1.0) < 1e-3,             "k must sum to 1"
print("PASS  k >= 0  and  sum(k) = 1  (simplex constraint satisfied)")

# u-subproblem: recovered pixels in [0, 255]
print("\nRunning u-subproblem test (8x8 image, 3x3 kernel)...")
t0 = time.time()
k_test = _gaussian_kernel(3, 1.0)
u_true = np.random.rand(8, 8) * 200 + 20
f_test2, _ = make_synthetic_blur(u_true, kernel_size=3, sigma=1.0, noise_std=2.0)
u_hat = solve_u_subproblem(k_test, f_test2, lambda1=0.05, lambda2=0.02)
print(f"  solved in {time.time()-t0:.1f}s")
assert np.all(u_hat >= -1e-3),        "u must be >= 0"
assert np.all(u_hat <= 255 + 1e-3),   "u must be <= 255"
print("PASS  0 <= u <= 255  (pixel range constraint satisfied)")

# k-subproblem reduces data fidelity
print("\nChecking that k-subproblem does not increase data fidelity...")
k_wrong = _gaussian_kernel(3, 2.5)   # deliberately wrong sigma
fid_before = data_fidelity(u_test, k_wrong, f_test)
k_better   = solve_k_subproblem(u_test, f_test, kernel_size=3, mu=0.001)
fid_after  = data_fidelity(u_test, k_better, f_test)
assert fid_after <= fid_before + 1e-3
print(f"PASS  fidelity before={fid_before:.2f}, after={fid_after:.2f}  (k-step reduces fidelity)")


---
### Test summary


In [ ]:
print("=" * 50)
print("All unit tests passed.")
print("Every mathematical claim in the model is verified.")
print("=" * 50)


---
## 4. End-to-end smoke test

Synthetic pipeline:
1. Create a simple sharp test image (geometric shapes — known ground truth)
2. Apply known Gaussian blur + noise → blurry observation $\mathbf{f}$
3. Run alternating minimization on $\mathbf{f}$
4. Compare recovered $\hat{\mathbf{u}}$ to ground truth using PSNR
5. Visualize: sharp / blurry / recovered / estimated kernel

> **Note on image size:** the matrix $H_k$ is $N \times N$ — for a $32 \times 32$
> image that is $1024 \times 1024$. This is the prototype scale. Larger images
> require the FFT-ADMM approach (next step after this notebook).


In [ ]:
np.random.seed(42)

# -- 4a. Create sharp test image (32x32 — prototype scale)
SIZE = 32
u_sharp = np.zeros((SIZE, SIZE))
u_sharp[6:14, 6:14]   = 200.0   # bright square top-left
u_sharp[18:28, 10:22]  = 150.0  # rectangle bottom
u_sharp[8:24, 20:28]   = 100.0  # overlapping rectangle right
# Add a soft ramp so Tikhonov has smooth regions to work with
ramp = np.linspace(0, 40, SIZE)
u_sharp += ramp[np.newaxis, :]
u_sharp = np.clip(u_sharp, 0, 255)

print(f"Sharp image: shape={u_sharp.shape}, min={u_sharp.min():.1f}, max={u_sharp.max():.1f}")


In [ ]:
# -- 4b. Apply synthetic blur
BLUR_SIGMA  = 1.2
NOISE_STD   = 3.0

f_obs, k_true = make_synthetic_blur(
    u_sharp, kernel_size=KERNEL_SIZE, sigma=BLUR_SIGMA, noise_std=NOISE_STD
)

psnr_blurry = psnr(u_sharp, f_obs)
print(f"True kernel sigma:  {BLUR_SIGMA}")
print(f"True kernel sum:    {k_true.sum():.6f}  (should be 1.0)")
print(f"Blurry image PSNR:  {psnr_blurry:.2f} dB  (baseline to beat)")


In [ ]:
# -- 4c. Run alternating minimization
print("Starting alternating minimization...")
print()

result = alternating_minimization(
    f          = f_obs,
    kernel_size = KERNEL_SIZE,
    lambda1    = LAMBDA1,
    lambda2    = LAMBDA2,
    mu         = MU,
    max_iter   = MAX_ITER,
    tol_k      = TOL,
    tol_u      = TOL,
    verbose    = True,
)

u_hat = result["u_hat"]
k_hat = result["k_hat"]
history = result["history"]

psnr_recovered = psnr(u_sharp, u_hat)
print()
print(f"Blurry PSNR:     {psnr_blurry:.2f} dB")
print(f"Recovered PSNR:  {psnr_recovered:.2f} dB  ({'improvement' if psnr_recovered > psnr_blurry else 'regression'})")
print()
print(f"Estimated kernel sum: {k_hat.sum():.6f}  (should be ~1.0)")
print(f"Estimated kernel min: {k_hat.min():.6f}  (should be >= 0)")


In [ ]:
# -- 4d. Visualize results
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

axes[0].imshow(u_sharp, cmap='gray', vmin=0, vmax=255)
axes[0].set_title(f'Sharp (ground truth)', fontsize=11)
axes[0].axis('off')

axes[1].imshow(f_obs, cmap='gray', vmin=0, vmax=255)
axes[1].set_title(f'Blurry observation\nPSNR = {psnr_blurry:.1f} dB', fontsize=11)
axes[1].axis('off')

axes[2].imshow(np.clip(u_hat, 0, 255), cmap='gray', vmin=0, vmax=255)
axes[2].set_title(f'Recovered u*\nPSNR = {psnr_recovered:.1f} dB', fontsize=11)
axes[2].axis('off')

axes[3].imshow(k_hat, cmap='hot', interpolation='nearest')
axes[3].set_title(f'Estimated kernel k*\n(true: Gaussian σ={BLUR_SIGMA})', fontsize=11)
axes[3].axis('off')

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/deconvolution_result.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved.")


In [ ]:
# -- 4e. Plot objective convergence
iters = range(1, len(history) + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: all four terms
for key, label, ls in [
    ("total",                   "Total J(k,u)",  '-'),
    ("data_fidelity",           "Data fidelity", '--'),
    ("tv_regularization",       "TV (λ₁)",       '-.'),
    ("tikhonov_regularization", "Tikhonov (λ₂)", ':'),
]:
    axes[0].plot(iters, [h[key] for h in history], ls, label=label, linewidth=2)
axes[0].set_xlabel("Iteration")
axes[0].set_ylabel("Objective value")
axes[0].set_title("Objective terms per iteration")
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Right: kernel L1 term alone
axes[1].plot(iters, [h["kernel_l1"] for h in history], 'o-', color='coral', linewidth=2)
axes[1].set_xlabel("Iteration")
axes[1].set_ylabel("μ ||k||₁")
axes[1].set_title("Kernel sparsity term")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/convergence_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print("Convergence plot saved.")


In [ ]:
# -- 4f. Kernel comparison: true vs estimated
fig, axes = plt.subplots(1, 3, figsize=(10, 3))

im0 = axes[0].imshow(k_true, cmap='hot', interpolation='nearest')
axes[0].set_title(f'True kernel\n(Gaussian σ={BLUR_SIGMA})')
plt.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(k_hat, cmap='hot', interpolation='nearest')
axes[1].set_title(f'Estimated kernel k*\n(sum={k_hat.sum():.4f})')
plt.colorbar(im1, ax=axes[1], fraction=0.046)

diff = np.abs(k_true - k_hat)
im2 = axes[2].imshow(diff, cmap='Blues', interpolation='nearest')
axes[2].set_title(f'|k_true - k*|\n(max error={diff.max():.4f})')
plt.colorbar(im2, ax=axes[2], fraction=0.046)

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/kernel_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Kernel comparison saved.")


---
## Summary

| Component | Math | Code |
|---|---|---|
| Gradient operator | $D\mathbf{u}$ | `gradient()` |
| TV regularization | $\|\nabla\mathbf{u}\|_1$ | `tv_norm()`, `cp.tv()` |
| Tikhonov regularization | $\|D\mathbf{u}\|_2^2$ | `tikhonov_norm_sq()` |
| Blur operator | $H\mathbf{u} = k * \mathbf{u}$ | `apply_blur()` |
| LASSO identity | $A_\mathbf{u} k = k * \mathbf{u}$ | `build_Au()` |
| Data fidelity | $\|H\mathbf{u} - \mathbf{f}\|_2^2$ | `data_fidelity()` |
| k-subproblem | constrained LASSO | `solve_k_subproblem()` |
| u-subproblem | TV + Tikhonov deconv | `solve_u_subproblem()` |
| Main loop | alternating minimization | `alternating_minimization()` |

**Next steps:** replace `build_Au_from_k` with FFT-based solve for larger images.
